Este Notebook sirve para generar grafos basados en significados de los contenidos de las noticias.

En el Notebook 03 se mide lo superficial con TF-IDF (palabras que aparecen, la frecuencia con la que aparecen, etc).


En este se usa SBERT (puede medir si dos dominios hablan de lo mismo, aunque usen palabras distintas)

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Ruta del proyecto en Google Drive
PROJECT_DIR = "/content/drive/MyDrive/TFG-FakeNewsNet"

# Me muevo a la carpeta principal del proyecto
%cd $PROJECT_DIR

!ls

/content/drive/MyDrive/TFG-FakeNewsNet
data  models  notebooks  README.md  requirements.txt  results


In [12]:
!pip install -q sentence-transformers  # Para SBERT
from sentence_transformers import SentenceTransformer

import pandas as pd
import numpy as np
from tqdm import tqdm
import joblib
import os

from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Cargo el dataset generado en el Notebook 02

df = pd.read_csv("data/noticias_preproc.csv")

print("Shape:", df.shape)
display(df.head(3))
display(df.info())

Shape: (44246, 15)


,title,text,subject,date,label,text_clean,n_chars,n_words,avg_word_len,n_exclam,n_question,n_digits,n_upper_words,url_count,has_url
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1,21st century wire say ben stein reputable prof...,1028,171,6.011696,0,0,7,12,0,0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0,washington reuter president donald trump remov...,4820,771,6.251621,0,0,6,19,0,0
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0,reuter puerto rico governor ricardo rossello s...,1848,304,6.078947,0,0,0,4,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44246 entries, 0 to 44245
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   title          44246 non-null  object 
 1   text           44246 non-null  object 
 2   subject        44246 non-null  object 
 3   date           44246 non-null  object 
 4   label          44246 non-null  int64  
 5   text_clean     44161 non-null  object 
 6   n_chars        44246 non-null  int64  
 7   n_words        44246 non-null  int64  
 8   avg_word_len   44246 non-null  float64
 9   n_exclam       44246 non-null  int64  
 10  n_question     44246 non-null  int64  
 11  n_digits       44246 non-null  int64  
 12  n_upper_words  44246 non-null  int64  
 13  url_count      44246 non-null  int64  
 14  has_url        44246 non-null  int64  
dtypes: float64(1), int64(9), object(5)
memory usage: 5.1+ MB


None

# Verificar TF-IDF

In [5]:
tfidf = joblib.load("models/tfidf_vectorizer.joblib")
print("TF-IDF cargado correctamente")


TF-IDF cargado correctamente


# Creación modelo SBERT

In [6]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # Modelo de SBERT
encoder = SentenceTransformer(MODEL_NAME)   # Convertir texto en embeddings

print("Modelo SBERT cargado:", MODEL_NAME)
print("Dimensión de embedding:", encoder.get_sentence_embedding_dimension())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo SBERT cargado: sentence-transformers/all-MiniLM-L6-v2
Dimensión de embedding: 384


# Generación embeddings para cada noticia

In [7]:
EMB_PATH = "data/embeddings_sbert.npy"

if os.path.exists(EMB_PATH):
    print("Embeddings ya existen.")
    embeddings = np.load(EMB_PATH)
    print("Embeddings cargados:", embeddings.shape)
    SKIP_EMBEDDINGS = True
else:
    print("No existen embeddings. Se calculan a continuación.")
    SKIP_EMBEDDINGS = False


Embeddings ya existen.
Embeddings cargados: (44246, 384)


In [8]:
if not SKIP_EMBEDDINGS:
    texts = df["text_clean"].astype(str).tolist()  # Convertir columna "text_clean" en una lista (la cual tendrá las noticias del dataset)

    embeddings = encoder.encode(     # Convertir las noticias en embeddings (vectores semánticos)
        texts,
        batch_size=128,    # Permite procesar 128 textos a la vez
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True    # Normalizar cada embedding (reduce ruido, comparación de dominios más estable)
    )

    print("Embeddings generados:", embeddings.shape)

else:
    print("Saltando cálculo. Embeddings ya cargados en memoria.")

Saltando cálculo. Embeddings ya cargados en memoria.


# Guardado de resultados

In [9]:
os.makedirs("data", exist_ok=True)
np.save(EMB_PATH, embeddings)

print("Embeddings guardados en", EMB_PATH)


Embeddings guardados en data/embeddings_sbert.npy


In [10]:
df["embed"] = list(embeddings)

domain_series = (
    df.groupby("subject")["embed"]   # Se agrupa las noticias por dominio
      .apply(lambda x: np.mean(np.vstack(x), axis=0))   # Se representa el dominio completo
)

domain_embeddings = np.vstack(domain_series.values)   # Convertir a matriz
domains = domain_series.index.tolist()

print("Subjects:", domains)
print("Shape embeddings por subject:", domain_embeddings.shape)


Subjects: ['Government News', 'Middle-east', 'News', 'US_News', 'left-news', 'politics', 'politicsNews', 'worldnews']
Shape embeddings por subject: (8, 384)


# Creación matriz de similitud coseno de dominios

In [14]:
sim_matrix = cosine_similarity(domain_embeddings)  # Calcula la matriz  de similitud coseno entre dominios

print("Shape sim_matrix:", sim_matrix.shape)


Shape sim_matrix: (8, 8)


# Generación aristas SBERT

Creo conexiones entre los nodos que se parecen semánticamente

In [15]:
edges = []

thr = 0.45  # Umbral de similitud mínima

for i, d1 in enumerate(domains):
    for j, d2 in enumerate(domains):
        if j <= i:    # Evita duplicados y que se compare un dominio consigo mismo
            continue
        weight = sim_matrix[i, j]
        if weight >= thr:  # Se crea la arista si supera el umbral
            edges.append((d1, d2, weight))

edges_df = pd.DataFrame(edges, columns=["source", "target", "weight"])
edges_df


,source,target,weight
0,Government News,Middle-east,0.874459
1,Government News,News,0.925310
2,Government News,US_News,0.874619
3,Government News,left-news,0.966387
4,Government News,politics,0.967762
5,Government News,politicsNews,0.901811
6,Government News,worldnews,0.742381
7,Middle-east,News,0.853291
8,Middle-east,US_News,0.999993
9,Middle-east,left-news,0.878940


# Guardado de los resultados

In [16]:
os.makedirs("results", exist_ok=True)
edges_df.to_csv("results/content_edges_sbert.csv", index=False)

print("Aristas SBERT guardadas en results/content_edges_sbert.csv")


Aristas SBERT guardadas en results/content_edges_sbert.csv


In [17]:
encoder.save("models/sbert_encoder")
print("Encoder SBERT guardado en models/sbert_encoder/")


Encoder SBERT guardado en models/sbert_encoder/
